# 05 — Pipeline development: `rivercast-data-ops` (Phase 9)

Develops and inspects the `rivercast-data-ops` KFP pipeline defined in
`pipelines/data_ops_pipeline.py`. The pipeline wires the Phase 8 components
(`fetch`, `transform`, `validate`, `monitor`, `forecast`) plus the new
Phase 9 delayed-metrics join into one hourly DAG:

```text
fetch (per station) -> transform -> validate -> [gate] -> forecast (per horizon)
                                          |
                                          +-> monitor
                                          +-> join-matured-predictions + delayed metrics
```

This notebook follows the plan's development loop: call each component
function directly first, then compile the pipeline to KFP YAML and inspect
its structure. The authoritative pipeline definition lives in
`pipelines/data_ops_pipeline.py`; this notebook only calls it and displays
results (CLAUDE.md rule 17).

> RiverCast is **educational**. Water level is relative to the local gauge
> zero — not river depth — and these forecasts must never inform real-world
> decisions.

In [ ]:
from datetime import UTC, datetime, timedelta
from pathlib import Path

from mlflow.client import MlflowClient

from components.common import open_store, read_json
from components.fetch.component import run as fetch_run
from components.forecast.component import run as forecast_run
from components.monitor.component import run as monitor_run
from components.transform.component import run as transform_run
from components.validate.component import run as validate_run
from rivercast.config import load_config
from rivercast.contracts.hourly import HourlyObservation
from rivercast.contracts.predictions import PredictionRecord
from rivercast.envcheck import find_lab_root
from rivercast.models.local_pipeline import run_training
from rivercast.models.registry import assign_challenger, get_champion, register_candidate
from rivercast.models.tracking import log_training_run, resolve_tracking_uri
from rivercast.processing.delayed_metrics import calculate_delayed_metrics, join_matured_predictions

LAB_ROOT = find_lab_root(Path.cwd())
config = load_config(LAB_ROOT / "configs" / "local.yaml")
FIXTURE_DIR = LAB_ROOT / "data_fixtures" / "pegelonline"

WINDOW_START = datetime(2024, 8, 1, tzinfo=UTC)
WINDOW_END = datetime(2024, 8, 8, tzinfo=UTC)

print(f"stations: {[s.name for s in config.stations]}")
print(f"horizons: {config.horizons_hours}")

## Step 1 — fetch every station

One `components.fetch` call per configured station, exactly what the
pipeline's `ParallelFor` loop does. Fetches are idempotent by content (raw
data is immutable, CLAUDE.md rule 9) — running this cell twice does not
duplicate archived payloads.

In [ ]:
for station in config.stations:
    assert station.uuid is not None
    result = fetch_run(
        config_path=LAB_ROOT / "configs" / "local.yaml",
        lab_root=LAB_ROOT,
        station_uuid=station.uuid,
        parameter=config.source.parameter,
        start=WINDOW_START,
        end=WINDOW_END,
        fixture_dir=FIXTURE_DIR,
    )
    print(f"{station.name:<10} status={result.status}  measurements={result.metadata.get('measurement_count')}")
    assert result.status == "ok"

## Step 2 — transform, then validate

`transform` builds bronze → silver hourly grid → gold training dataset in
one step. `validate` runs the quality-check battery over the silver window,
**including the target-station freshness check** — the gate the pipeline's
`dsl.If` branches on before issuing any forecast (PLAN.md Phase 9 schedule:
"if source data is not fresh enough: do not issue a forecast").

In [ ]:
CONFIG_PATH = LAB_ROOT / "configs" / "local.yaml"

transform_result = transform_run(
    config_path=CONFIG_PATH, lab_root=LAB_ROOT, start=WINDOW_START, end=WINDOW_END
)
print(f"transform status: {transform_result.status}")
assert transform_result.status == "ok"
silver_key = transform_result.output_keys[0]
dataset_id = transform_result.metadata["dataset_id"]
dataset_short_id = dataset_id.removeprefix("sha256:")[:12]
print(f"dataset_id: {dataset_id}")

validate_result = validate_run(
    config_path=CONFIG_PATH, lab_root=LAB_ROOT, silver_key=silver_key, now_utc=WINDOW_END
)
print(f"validate status: {validate_result.status} (passed={validate_result.metadata.get('passed')})")
assert validate_result.status == "ok"

## The freshness gate, demonstrated

Re-running validation with `now_utc` far past the fixture window shows
exactly what the pipeline's `dsl.If(validate.output == "ok")` branch does in
that case: `status` becomes `"failed"`, and the pipeline skips
`forecast_task` entirely for this run — the champion is never called with
stale input, and no forecast is fabricated.

In [ ]:
far_future = WINDOW_END + timedelta(days=30)
stale_result = validate_run(
    config_path=CONFIG_PATH, lab_root=LAB_ROOT, silver_key=silver_key, now_utc=far_future
)
print(f"validate status with now={far_future.date()}: {stale_result.status}")
print(f"errors: {stale_result.metadata.get('errors')}")
assert stale_result.status == "failed"
should_forecast = stale_result.status == "ok"
print(f"pipeline would issue a forecast: {should_forecast}")
assert not should_forecast

## Step 3 — monitor

Runs independently of the freshness gate — it reports on the data itself,
not on whether a forecast was issued.

In [ ]:
monitor_result = monitor_run(
    config_path=CONFIG_PATH, lab_root=LAB_ROOT, silver_key=silver_key, now_utc=WINDOW_END
)
print(f"monitor status: {monitor_result.status}")
print(f"row_count={monitor_result.metadata['row_count']}  missing_stations={monitor_result.metadata['missing_station_count']}")

## Step 4 — a champion, so forecast has something to load

`forecast` loads the current champion by MLflow alias. Train, register, and
promote a 6h ridge candidate directly through the Phase 6/7 functions (the
same ones `rivercast train --promote` uses) so this notebook can demonstrate
a real forecast next.

In [ ]:
train_result = run_training(
    config=config,
    fixture_dir=FIXTURE_DIR,
    horizon_hours=6,
    model_name="ridge",
    models_dir=LAB_ROOT / "models" / "local",
    seed=42,
)
logged_run = log_training_run(config, LAB_ROOT, train_result)
tracking_uri = resolve_tracking_uri(config, LAB_ROOT)
client = MlflowClient(tracking_uri=tracking_uri)
model_version = register_candidate(client, "rivercast-kaub-6h", logged_run)
assign_challenger(client, "rivercast-kaub-6h", model_version)
client.set_registered_model_alias("rivercast-kaub-6h", "champion", model_version.version)

champion = get_champion(client, "rivercast-kaub-6h")
print(f"champion: rivercast-kaub-6h v{champion.version}")
assert champion is not None

## Step 5 — issue and persist a forecast

`forecast` loads the champion, scores one feature row, and writes a
`PredictionRecord` (PLAN.md Phase 9 schema) to the `predictions` object-store
zone — exactly what the pipeline's per-horizon `ParallelFor` inside the
`dsl.If` gate does.

In [ ]:
import io

import pandas as pd

dataset_key = f"gold/training/dataset_id={dataset_short_id}/dataset.parquet"
store = open_store(config, LAB_ROOT)
dataset = pd.read_parquet(io.BytesIO(store.get_bytes(dataset_key)))
feature_columns = [
    c for c in dataset.columns if not c.startswith("target_level_") and c != "issue_time_utc"
]
latest_row = dataset.iloc[-1][feature_columns].fillna(0.0)
# missing_<station> columns are logged as int64 in the model's inferred
# signature (0/1 flags only); keep them as Python int, not float, or
# mlflow's strict schema enforcement rejects the prediction request.
features = {
    col: int(latest_row[col]) if col.startswith("missing_") else float(latest_row[col])
    for col in feature_columns
}

issue_time = WINDOW_START + timedelta(hours=24)
forecast_result = forecast_run(
    config_path=CONFIG_PATH,
    lab_root=LAB_ROOT,
    horizon_hours=6,
    issue_time=issue_time,
    features=features,
)
print(f"forecast status: {forecast_result.status}")
print(f"prediction_cm: {forecast_result.metadata.get('prediction_cm'):.2f}")
print(f"model_alias: {forecast_result.metadata.get('model_alias')}")
assert forecast_result.status == "ok"


## Step 6 — join matured predictions and calculate delayed metrics

Once `target_time_utc` has passed and a real observation exists,
`join_matured_predictions` supplies the actual value and error; predictions
still in the future join to `actual_cm=None` rather than a fabricated value
(rule 13: never fabricate). This is the pipeline's
`join-matured-predictions` / `calculate-delayed-metrics` step.

In [ ]:
prediction_key = forecast_result.output_keys[0]
prediction = PredictionRecord.model_validate(read_json(store, prediction_key))
hourly = [HourlyObservation.model_validate(row) for row in read_json(store, silver_key)]

now_after_maturity = issue_time + timedelta(hours=7)  # 1h past target_time_utc (issue_time + 6h)
matured = join_matured_predictions([prediction], hourly, now_utc=now_after_maturity)[0]

print(f"matured: {matured.is_matured}")
print(
    f"prediction_cm={prediction.prediction_cm:.2f}  "
    f"actual_cm={matured.actual_cm:.2f}  error_cm={matured.error_cm:.2f}"
)
assert matured.is_matured

metrics = calculate_delayed_metrics([matured], horizon_hours=6)
print(f"delayed MAE (n={metrics.n_matured}): {metrics.mae_cm:.2f} cm")


## Compile the pipeline to KFP YAML

The authoritative pipeline definition (`pipelines/data_ops_pipeline.py`)
compiles to a static DAG independent of any of the runtime values above —
`compiler.Compiler().compile(...)` inspects the pipeline's Python structure,
not live data.

In [ ]:
import yaml
from kfp import compiler

from pipelines.data_ops_pipeline import rivercast_data_ops_pipeline

compiled_path = LAB_ROOT / "pipelines" / "compiled" / "rivercast-data-ops.yaml"
compiler.Compiler().compile(rivercast_data_ops_pipeline, str(compiled_path))
print(f"compiled to {compiled_path} ({compiled_path.stat().st_size} bytes)")

doc = yaml.safe_load(compiled_path.read_text(encoding="utf-8"))
print(f"pipeline name: {doc['pipelineInfo']['name']}")
print(f"root tasks: {sorted(doc['root']['dag']['tasks'].keys())}")


## Local execution: environment limitation

`kfp.local.SubprocessRunner` is the plan's recommended way to run compatible
components locally in the workbench (PLAN.md Phase 8/9 development loop).
On this development machine, KFP's subprocess runner invokes the interpreter
through a POSIX shell that mis-resolves a Windows-style `.venv` interpreter
path — this reproduces even for a trivial two-argument component with no
RiverCast code involved, so it is an environment limitation, not a defect in
this pipeline. The pipeline's **structure** is fully verified above (every
component function proven directly, the compiled YAML inspected); full
`SubprocessRunner` execution should be re-verified in the actual Linux
OpenShift AI JupyterLab workbench, where a trainee would run this next
(`kfp.local.init(runner=kfp.local.SubprocessRunner())` then call
`rivercast_data_ops_pipeline(...)` directly).

## Conclusion

Every pipeline step — fetch, transform, validate (with the freshness gate),
monitor, forecast, and the delayed-metrics join — works both as a directly
callable function and as a compiled KFP DAG node. The freshness gate is a
real `dsl.If` branch on `validate`'s typed output, not a placeholder: a
stale window measurably skips forecasting. Next: Phase 10 builds the
`rivercast-model` pipeline (scheduled retraining with champion/challenger
promotion gates).